In [1]:
%%writefile solution.py
from __future__ import annotations
import json
import pickle
from pathlib import Path
import numpy as np
import pandas as pd
from common import load_model, predict_labels, save_model, vectorize_texts, softmax

def onehoty(y):
    Y = []
    for yy in y:
        aux = [0, 0, 0, 0]
        aux[yy] = 1
        Y.append(aux)
    return np.array(Y)

def cross_entropy(P, labels):
    total = 0.0
    for i in range(len(labels)):
        true = P[i, labels[i]]
        total = total + (-np.log(true + 1e-12))
    return total / len(labels)

def adapt(
    phase2_train_csv,
    pretrained_model_dir,
    output_dir,
    seed,
):
    phase2_train_csv = Path(phase2_train_csv)
    pretrained_model_dir = Path(pretrained_model_dir)
    output_dir = Path(output_dir)

    output_dir.mkdir(parents=True, exist_ok=True)

    # Citim datele doar pentru a verifica faptul că fișierul există și este valid.
    train_data = pd.read_csv(phase2_train_csv)
    X = vectorize_texts(train_data['text'])
    y = train_data['answer'].values.astype(int)
    Y = onehoty(y)
    

    model = load_model(pretrained_model_dir)

    with open(pretrained_model_dir / "state.pkl", "rb") as fin:
         pretrained_state = pickle.load(fin)
    W = pretrained_state["W"].copy()
    b = pretrained_state["b"].copy()

    lr = 1e-2
    epochs = 2000
    for epoch in range(epochs):
        P = softmax(X @ W + b)
        loss = cross_entropy(P, y)
        W_grad = X.T @ ((P - Y) / len(y))
        b_grad = (P-Y).mean(axis=0)

        W = W - W_grad*lr
        b = b - b_grad*lr
        
        
    save_model(output_dir, {"W": W, "b": b}, metadata={"seed": int(seed)})
    manifest = {"format_version": 1, "model_files": ["state.pkl", "meta.json"]}
    manifest_path = output_dir / "manifest.json"
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    return str(manifest_path)

def predict(
    test_csv,
    model_dir,
):
    test_csv = Path(test_csv)
    model_dir = Path(model_dir)

    test = pd.read_csv(test_csv)

    model = load_model(model_dir)
    X = vectorize_texts(test["text"])
    predictions = predict_labels(model, X)

    return pd.DataFrame(
        {
            "datapointID": test["datapointID"].astype(str),
            "subtaskID": test["subtaskID"].astype(int),
            "answer": predictions,
        },
        columns=["datapointID", "subtaskID", "answer"],
    )

Overwriting solution.py


In [2]:
import importlib, solution
importlib.reload(solution)
from solution import adapt, predict

In [3]:
import zipfile
with zipfile.ZipFile('submission.zip', 'w') as z:
   z.write('solution.py')
   z.write('common.py')